In [1]:
import pandas as pd
import zipfile
import os
import string
import nltk
from nltk.corpus import stopwords

# Google Drive bağlantısı
from google.colab import drive
drive.mount('/content/drive')

# ZIP dosyasını çıkarma
zip_dosya_yolu = '/content/drive/MyDrive/turkish-text-data/furki.zip'
cikarma_dizini = '/content/drive/MyDrive/turkish-text-data/cikarilan/'
os.makedirs(cikarma_dizini, exist_ok=True)

with zipfile.ZipFile(zip_dosya_yolu, 'r') as zip_ref:
    zip_ref.extractall(cikarma_dizini)

# Dosya yolları
pozitif_dosya_yolu = os.path.join(cikarma_dizini, 'furki.pos')
negatif_dosya_yolu = os.path.join(cikarma_dizini, 'furki.neg')

# Veri yükleme
pozitif_df = pd.read_csv(pozitif_dosya_yolu, delimiter='\t', header=None, names=['review'])
pozitif_df['label'] = 'positive'

negatif_df = pd.read_csv(negatif_dosya_yolu, delimiter='\t', header=None, names=['review'])
negatif_df['label'] = 'negative'

# Veri birleştirme
data = pd.concat([pozitif_df, negatif_df], ignore_index=True)

# Stopwords ve noktalama
nltk.download('stopwords')
etkisiz = stopwords.words('turkish')
noktalama = string.punctuation

# Veri temizleme
def temizle(metin):
    metin = ''.join([char for char in metin if char not in noktalama])  # Noktalama kaldırma
    metin = ' '.join([word for word in metin.split() if word.lower() not in etkisiz])  # Stopwords kaldırma
    return metin

data['review'] = data['review'].apply(temizle)

# Temizlenmiş veriyi kaydetme
data.to_csv('./cleaned.csv', index=False)


Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
# Dengeli veri seti oluşturma
positive_samples = data[data['label'] == 'positive'].sample(n=499, random_state=42)  # Negatiflere eşit örnek alınıyor
negative_samples = data[data['label'] == 'negative']
balanced_data = pd.concat([positive_samples, negative_samples])

# Train-test ayırma
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    balanced_data['review'].values.astype('U'),
    balanced_data['label'].values.astype('U'),
    test_size=0.1, random_state=42
)


In [3]:
# Sayma ve TF-IDF vektörlerini çıkarma
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

count_vect = CountVectorizer()
x_train_counts = count_vect.fit_transform(x_train)

tfidf_transformer = TfidfTransformer()
x_train_tfidf = tfidf_transformer.fit_transform(x_train_counts)

# Test verisini dönüştürme
x_test_counts = count_vect.transform(x_test)
x_test_tfidf = tfidf_transformer.transform(x_test_counts)


In [4]:
# Multinomial Naive Bayes modeli
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB().fit(x_train_tfidf, y_train)

# Tahminler
y_pred = clf.predict(x_test_tfidf)

# Performans ölçümleri
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['negative', 'positive']))


Accuracy: 0.83

Classification Report:
               precision    recall  f1-score   support

    negative       0.90      0.79      0.84        58
    positive       0.76      0.88      0.81        42

    accuracy                           0.83       100
   macro avg       0.83      0.84      0.83       100
weighted avg       0.84      0.83      0.83       100



In [5]:
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression(class_weight='balanced', random_state=42).fit(x_train_tfidf, y_train)

# Logistic Regression tahminler
y_pred_lr = clf_lr.predict(x_test_tfidf)

# Performans ölçümleri
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nLogistic Regression Classification Report:\n", classification_report(y_test, y_pred_lr, target_names=['negative', 'positive']))


Logistic Regression Accuracy: 0.84

Logistic Regression Classification Report:
               precision    recall  f1-score   support

    negative       0.84      0.90      0.87        58
    positive       0.84      0.76      0.80        42

    accuracy                           0.84       100
   macro avg       0.84      0.83      0.83       100
weighted avg       0.84      0.84      0.84       100

